In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
import unicodedata
from sklearn.model_selection import train_test_split
import os

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
print(os.getcwd())

c:\Users\hp\AppData\Local\Programs\Microsoft VS Code


In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Text Cleaning

In [5]:
def clean_text(text):
     
    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)   
    
    text = text.encode("ascii", errors="ignore").decode()
    
    text = re.sub(r"http\S+" ," ",text)
    
    text = re.sub(r"<.*?>", " ", text)
    
    text = re.sub(r"\bbr\b", " ", text)

    text = re.sub(r"[^a-zA-Z\s]" , " " , text)

    text = re.sub(r"\s+"," ",text).strip()
    
    return text

In [6]:
df["clean_review"] = df["review"].apply(clean_text)

In [7]:
print(df[["review", "clean_review"]].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  one of the other reviewers has mentioned that ...  
1  a wonderful little production the filming tech...  
2  i thought this was a wonderful way to spend ti...  
3  basically there s a family where a little boy ...  
4  petter mattei s love in the time of money is a...  


# Tokenization


In [8]:
def tokenize(text):
    tokens=text.split()

    return tokens

In [9]:
df["tokens"] = df["clean_review"].apply(tokenize)

In [10]:
print(df[["clean_review" , "tokens"]].head())

                                        clean_review  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production the filming tech...   
2  i thought this was a wonderful way to spend ti...   
3  basically there s a family where a little boy ...   
4  petter mattei s love in the time of money is a...   

                                              tokens  
0  [one, of, the, other, reviewers, has, mentione...  
1  [a, wonderful, little, production, the, filmin...  
2  [i, thought, this, was, a, wonderful, way, to,...  
3  [basically, there, s, a, family, where, a, lit...  
4  [petter, mattei, s, love, in, the, time, of, m...  


# Negation Handling

In [11]:
def handle_negation(tokens):
    result = []
    negate = False

    for word in tokens:
        if word in ["not", "no", "never"]:
            negate = True
            continue
        if negate:
            result.append("NOT_" + word)
        else:
            result.append(word)
    return result

In [12]:
df["tokens_neg"] = df["tokens"].apply(handle_negation)

In [13]:
print(df[["clean_review", "tokens_neg"]].head())

                                        clean_review  \
0  one of the other reviewers has mentioned that ...   
1  a wonderful little production the filming tech...   
2  i thought this was a wonderful way to spend ti...   
3  basically there s a family where a little boy ...   
4  petter mattei s love in the time of money is a...   

                                          tokens_neg  
0  [one, of, the, other, reviewers, has, mentione...  
1  [a, wonderful, little, production, the, filmin...  
2  [i, thought, this, was, a, wonderful, way, to,...  
3  [basically, there, s, a, family, where, a, lit...  
4  [petter, mattei, s, love, in, the, time, of, m...  


# Stopwords Removal


In [ ]:
stop_words = set(stopwords.words('english'))
NEGATION_WORDS = {"not", "no", "never", "nor", "neither"}
stop_words = stop_words - NEGATION_WORDS
print(stop_words)

In [16]:
def remove_stopwords(tokens):

    filtered = []
     
    for word in tokens:
       if word not in stop_words:
        filtered.append(word)

    return filtered

In [17]:
df["not_stopwords"] = df["tokens"].apply(remove_stopwords)

In [18]:
print(df[["tokens" , "not_stopwords"]].head())

                                              tokens  \
0  [one, of, the, other, reviewers, has, mentione...   
1  [a, wonderful, little, production, the, filmin...   
2  [i, thought, this, was, a, wonderful, way, to,...   
3  [basically, there, s, a, family, where, a, lit...   
4  [petter, mattei, s, love, in, the, time, of, m...   

                                       not_stopwords  
0  [one, reviewers, mentioned, watching, oz, epis...  
1  [wonderful, little, production, filming, techn...  
2  [thought, wonderful, way, spend, time, hot, su...  
3  [basically, family, little, boy, jake, thinks,...  
4  [petter, mattei, love, time, money, visually, ...  


# Train-Test Split

In [19]:
X = df["not_stopwords"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {(X_train).shape}")
print(f"Test size: {(X_test).shape}")
print(f"Train size: {(y_train).shape}")
print(f"Test size: {(y_test).shape}")

Train size: (40000,)
Test size: (10000,)
Train size: (40000,)
Test size: (10000,)


In [20]:
df.to_csv("cleand date")

GloVe embedding


In [1]:
import ast
import numpy as np
import pandas as pd
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [2]:
POSITIVE_WORDS = {
    "good", "great", "excellent", "amazing", "wonderful", "love", "loved",
    "best", "fantastic", "beautiful", "superb", "perfect", "enjoy", "enjoyed",
    "brilliant", "awesome", "favorite", "fun", "happy", "nice",
}
NEGATIVE_WORDS = {
    "bad", "terrible", "awful", "worst", "hate", "hated", "boring", "poor",
    "waste", "worse", "horrible", "disappointing", "stupid", "dull",
    "annoying", "ridiculous", "fail", "failed", "ugly", "lame",
}
NEGATION_WORDS = {"not", "no", "never", "nor", "neither"}

In [3]:
def sentence_to_vector(tokens, model, dim=300):
    """
    Average GloVe vector. Words right after a negation are down-weighted
    so the average isn't pulled toward their un-negated meaning.
    """
    vectors, weights = [], []
    after_neg = False
    for token in tokens:
        if token in NEGATION_WORDS:
            after_neg = True
            continue
        if token in model:
            vectors.append(model[token])
            weights.append(0.25 if after_neg else 1.0)
        after_neg = False
    if not vectors:
        return np.zeros(dim, dtype=np.float32)
    return np.average(np.asarray(vectors), axis=0,
                      weights=np.asarray(weights)).astype(np.float32)


In [4]:
def extra_features(tokens):
    """Eight cheap, hand-crafted sentiment features."""
    n = max(len(tokens), 1)
    pos  = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg  = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    negs = sum(1 for t in tokens if t in NEGATION_WORDS)
    neg_pos = neg_neg = 0
    flipped = False
    for t in tokens:
        if t in NEGATION_WORDS:
            flipped = True
            continue
        if flipped and t in POSITIVE_WORDS: neg_pos += 1
        if flipped and t in NEGATIVE_WORDS: neg_neg += 1
        flipped = False
    return np.array([
        np.log1p(n),                                # 1. log review length
        pos / n,                                    # 2. positive density
        neg / n,                                    # 3. negative density
        negs / n,                                   # 4. negation density
        neg_pos / n,                                # 5. negated-positive ("not good")
        neg_neg / n,                                # 6. negated-negative ("not bad")
        (pos - neg + neg_neg - neg_pos) / n,        # 7. flip-aware net polarity
        (pos + neg) / n,                            # 8. sentiment density
    ], dtype=np.float32)

In [5]:
def sentence_to_full_vector(tokens, model, dim=300):
    """Concatenate avg-GloVe (300) and hand-crafted features (8) -> 308."""
    return np.concatenate([
        sentence_to_vector(tokens, model, dim=dim),
        extra_features(tokens),
    ])

In [6]:
if __name__ == "__main__":
    print("Loading preprocessed dataset...")
    df = pd.read_csv("cleand date.csv")
    df["not_stopwords"] = df["not_stopwords"].apply(ast.literal_eval)

    print("Loading GloVe model...")
    embedding_model = api.load("glove-wiki-gigaword-300")

    print("Building 308-D embeddings...")
    tqdm.pandas(desc="Embed")
    X = np.stack(df["not_stopwords"].progress_apply(
        lambda t: sentence_to_full_vector(t, embedding_model)
    ).values)
    y = np.array(
        [1 if str(s).lower() == "positive" else 0 for s in df["sentiment"]],
        dtype=np.int64,
    )

    print(f"Shapes: X={X.shape}, y={y.shape}")

    print("Splitting train/test (stratified)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print("Saving .npy files...")
    np.save("X_train.npy", X_train)
    np.save("X_test.npy", X_test)
    np.save("y_train.npy", y_train)
    np.save("y_test.npy", y_test)
    np.save("X_embeddings.npy", X)
    np.save("y_labels.npy", y)
    print("Done!")

Loading preprocessed dataset...
Loading GloVe model...
Building 308-D embeddings...


Embed: 100%|██████████| 50000/50000 [00:59<00:00, 837.25it/s] 


Shapes: X=(50000, 308), y=(50000,)
Splitting train/test (stratified)...
Saving .npy files...
Done!


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time

# =========================================
# 1) Define Feed-Forward Neural Network
# =========================================
class SentimentNN(nn.Module):
    def __init__(self, input_dim):
        super(SentimentNN, self).__init__()
        # Input layer -> Hidden Layer 1
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.4)
        
        # Hidden Layer 1 -> Hidden Layer 2
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.3)
        
        # Hidden Layer 2 -> Hidden Layer 3
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.2)
        
        # Hidden Layer 3 -> Output Layer
        self.fc4 = nn.Linear(64, 1)
        
        # Activations
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout1(out)
        
        out = self.fc2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.dropout2(out)
        
        out = self.fc3(out)
        out = self.bn3(out)
        out = self.relu(out)
        out = self.dropout3(out)
        
        out = self.fc4(out)
        out = self.sigmoid(out)
        return out

def load_data():
    print("Loading data...")
    X_train = np.load("X_train.npy")
    y_train = np.load("y_train.npy")
    X_test = np.load("X_test.npy")
    y_test = np.load("y_test.npy")

    print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
    print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")
    return X_train, y_train, X_test, y_test

def main():
    # Setup Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load Data
    X_train, y_train, X_test, y_test = load_data()

    # Convert to Tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

    # Create DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    # Initialize model
    input_dimension = X_train.shape[1]  # Should be 308 now (300 GloVe + 8 features)
    model = SentimentNN(input_dimension).to(device)
    print(f"Model input dimension: {input_dimension}")

    # Optimization, Loss, and Scheduler
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    # Training Loop
    epochs = 40
    best_accuracy = 0.0
    print("\nStarting Training...")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(train_loader)
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']
        
        # Quick validation every epoch
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                predicted = (outputs > 0.5).float()
                val_preds.extend(predicted.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds) * 100
        
        # Save best model
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            torch.save(model.state_dict(), "sentiment_model.pth")
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Val Acc: {val_acc:.2f}%, Best: {best_accuracy:.2f}%, LR: {current_lr:.6f}, Time: {epoch_time:.2f}s")
        
        model.train()
        scheduler.step(epoch_loss)

    # Load best model for final evaluation
    model.load_state_dict(torch.load("sentiment_model.pth", map_location=device))
    model.eval()
    
    print("\nFinal Evaluation (Best Model)...")
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            predicted = (outputs > 0.5).float()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    print("\n--- Test Results ---")
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("=========================================")
    print(f"\nBest model saved to sentiment_model.pth")

if __name__ == "__main__":
    main()

Using device: cpu
Loading data...
Train shapes: X=(40000, 308), y=(40000,)
Test shapes: X=(10000, 308), y=(10000,)
Model input dimension: 308

Starting Training...
Epoch [1/60], Loss: 0.3968, Val Acc: 85.18%, Best: 85.18%, LR: 0.001000, Time: 2.97s
Epoch [2/60], Loss: 0.3465, Val Acc: 84.19%, Best: 85.18%, LR: 0.001000, Time: 4.78s
Epoch [3/60], Loss: 0.3361, Val Acc: 85.30%, Best: 85.30%, LR: 0.001000, Time: 4.92s
Epoch [4/60], Loss: 0.3248, Val Acc: 83.77%, Best: 85.30%, LR: 0.001000, Time: 4.09s
Epoch [5/60], Loss: 0.3203, Val Acc: 82.28%, Best: 85.30%, LR: 0.001000, Time: 4.52s
Epoch [6/60], Loss: 0.3106, Val Acc: 85.87%, Best: 85.87%, LR: 0.001000, Time: 4.73s
Epoch [7/60], Loss: 0.3047, Val Acc: 85.80%, Best: 85.87%, LR: 0.001000, Time: 4.36s
Epoch [8/60], Loss: 0.3006, Val Acc: 84.17%, Best: 85.87%, LR: 0.001000, Time: 3.97s
Epoch [9/60], Loss: 0.2931, Val Acc: 86.22%, Best: 86.22%, LR: 0.001000, Time: 4.57s
Epoch [10/60], Loss: 0.2887, Val Acc: 85.47%, Best: 86.22%, LR: 0.00100